# 📊 평가 실행

이 노트북은 학습된 모델들을 세 가지 트랙으로 평가합니다.

## 세 가지 평가 트랙

| 트랙 | 목적 | 측정 지표 |
|------|------|----------|
| **A. `prepared_diagnostics`** | 오프라인 홀드아웃 데이터로 진단 | 답변 정확도, 도구 일치, 근거 점수 |
| **B. `tau_episodes`** | 공식 τ-Knowledge 에피소드 실행 | task_success_rate, pass^k |
| **C. `retention`** | 기존 능력 보존 확인 | ARC-Challenge 정확도, retention_delta_pp |

### 핵심 구분

- **A는 실험실 진단**: 합성 단일 턴 정확도이며, 공식 τ 점수가 **아닙니다**.
- **B는 공식 평가**: 실제 에피소드를 실행하며, 배포된 모델 엔드포인트를 호출합니다.
- **C는 망각 테스트**: KB/RAG를 비활성화하고 일반 벤치마크를 실행합니다.

### 비교 변형 (Variants)

```
Base × {no_knowledge, rag}
LoRA × {no_knowledge, rag}
OSFT × {no_knowledge, rag}
```

모든 변형은 동일한 비즈니스 도구와 시뮬레이터 조건을 사용합니다.

> ⚠️ `tau_episodes` 트랙은 배포된 모델 엔드포인트가 필요합니다.  
> 엔드포인트가 없으면 `prepared_diagnostics`와 `retention`만 실행됩니다.

In [ ]:
"""환경 부트스트랩 — local과 workbench 모두 지원."""

import subprocess, sys, os
from pathlib import Path

# 프로젝트 루트 탐색
_nb_dir = Path.cwd()
_project_root = _nb_dir
for _p in [_nb_dir] + list(_nb_dir.parents):
    if (_p / "pyproject.toml").exists():
        _project_root = _p
        break

# 패키지 설치 확인 및 자동 설치
try:
    import rhoai_model_training_lab  # noqa: F401
    print("✅ rhoai_model_training_lab 패키지 확인됨")
except ImportError:
    print("📦 패키지 설치 중... (최초 1회)")
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-e", str(_project_root),
             "--extra-index-url", "https://pypi.org/simple/"],
            stdout=subprocess.DEVNULL,
        )
        print("✅ 설치 완료")
    except subprocess.CalledProcessError:
        # Red Hat Workbench 등 제한된 환경: sys.path fallback
        print("⚠️  pip install 실패 — sys.path fallback 사용")
        _src = str(_project_root / "src")
        if _src not in sys.path:
            sys.path.insert(0, _src)
        # 핵심 의존성만 설치 시도
        for _dep in ["pydantic", "python-dotenv", "pyyaml", "rich", "httpx"]:
            try:
                __import__(_dep.replace("-", "_"))
            except ImportError:
                subprocess.call(
                    [sys.executable, "-m", "pip", "install", _dep,
                     "--extra-index-url", "https://pypi.org/simple/"],
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                )
        import rhoai_model_training_lab  # noqa: F401
        print("✅ sys.path fallback 설정 완료")


In [ ]:
"""Run prepared_diagnostics — offline holdout evaluation."""

import os
import sys
import subprocess
from pathlib import Path

from rhoai_model_training_lab.config import load_env, load_eval_config, PROJECT_ROOT

load_env()

eval_config = load_eval_config()
output_dir = PROJECT_ROOT / eval_config["general"]["output_dir"]
output_dir.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("📊 트랙 A: prepared_diagnostics (오프라인 홀드아웃)")
print("=" * 70)

diag_config = eval_config["tracks"]["prepared_diagnostics"]
if not diag_config.get("enabled", True):
    print("⏭️ prepared_diagnostics 비활성화됨, 건너뜁니다.")
else:
    print(f"  홀드아웃 경로: {diag_config['holdout_path']}")
    print(f"  평가 유형: {diag_config['types']}")
    print(f"  matched_context: {diag_config.get('matched_context', False)}")
    print()

    # Run via CLI
    run_eval_script = PROJECT_ROOT / "scripts" / "run_eval.py"

    if run_eval_script.exists():
        cmd = [
            sys.executable, str(run_eval_script),
            "--track", "prepared_diagnostics",
            "--config", "configs/eval.yaml",
        ]
        print(f"  실행 명령: {' '.join(cmd)}")
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))

        if result.returncode == 0:
            print("\n✅ prepared_diagnostics 완료")
            print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
        else:
            print(f"\n❌ 실행 실패 (exit code: {result.returncode})")
            print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
    else:
        print("  ⚠️  run_eval.py 스크립트를 찾을 수 없습니다.")
        print("  수동 실행:")
        print("    python scripts/run_eval.py --track prepared_diagnostics --config configs/eval.yaml")

    print("\n  ℹ️  이 결과는 'τ-Knowledge 파생 실험실 진단'으로 표기됩니다.")
    print("     합성 단일 턴 정확도 ≠ 공식 τ pass rate")

In [ ]:
"""Run tau_episodes smoke test (5 episodes, 1 trial)."""

print("=" * 70)
print("📊 트랙 B: tau_episodes (공식 τ-Knowledge 에피소드)")
print("=" * 70)

tau_config = eval_config["tracks"]["tau_episodes"]
if not tau_config.get("enabled", True):
    print("⏭️ tau_episodes 비활성화됨, 건너뜁니다.")
else:
    smoke_limit = tau_config.get("smoke_limit", 5)
    smoke_trials = tau_config.get("smoke_trials", 1)

    print(f"  도메인: {tau_config.get('domain', 'banking_knowledge')}")
    print(f"  스모크 제한: {smoke_limit} 에피소드, {smoke_trials} 시행")
    print(f"  변형: {[v['name'] for v in tau_config.get('variants', [])]}")
    print()

    # Check if model endpoints are available
    base_endpoint = os.environ.get("BASE_SERVING_ENDPOINT", "")
    if not base_endpoint:
        print("⚠️  모델 서빙 엔드포인트가 설정되지 않았습니다.")
        print("   BASE_SERVING_ENDPOINT, LORA_SERVING_ENDPOINT, OSFT_SERVING_ENDPOINT를 설정하세요.")
        print("   스모크 테스트를 건너뜁니다.")
    else:
        run_eval_script = PROJECT_ROOT / "scripts" / "run_eval.py"
        if run_eval_script.exists():
            cmd = [
                sys.executable, str(run_eval_script),
                "--track", "tau_episodes",
                "--config", "configs/eval.yaml",
                "--limit", str(smoke_limit),
                "--trials", str(smoke_trials),
            ]
            print(f"  실행 명령: {' '.join(cmd)}")
            print("  (이 과정은 여러 분이 소요될 수 있습니다)\n")

            result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))

            if result.returncode == 0:
                print("\n✅ tau_episodes 스모크 테스트 완료")
                print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
            else:
                print(f"\n❌ 실행 실패 (exit code: {result.returncode})")
                print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
        else:
            print("  ⚠️  run_eval.py 스크립트 없음")
            print("  수동 실행:")
            print(f"    python scripts/run_eval.py --track tau_episodes --config configs/eval.yaml --limit {smoke_limit} --trials {smoke_trials}")

    print("\n  ℹ️  스모크 실행은 경로 검증용이며, 벤치마크 성능 결론이 아닙니다.")
    print("     pass^k는 반복 시행에서의 일관된 성공을 측정합니다 (pass@k와 다름).")

In [ ]:
"""Run retention benchmark (ARC-Challenge)."""

print("=" * 70)
print("📊 트랙 C: retention (기존 능력 보존)")
print("=" * 70)

retention_config = eval_config["tracks"]["retention"]
if not retention_config.get("enabled", True):
    print("⏭️ retention 비활성화됨, 건너뜁니다.")
else:
    benchmarks = retention_config.get("benchmarks", [])
    print(f"  벤치마크: {[b['name'] for b in benchmarks]}")
    print(f"  KB/RAG 비활성화: {retention_config.get('disable_kb_rag', True)}")
    print(f"  뱅킹 시스템 프롬프트 비활성화: {retention_config.get('disable_banking_system_prompt', True)}")
    print()

    for bench in benchmarks:
        print(f"\n  📝 {bench['name']}:")
        print(f"     데이터셋: {bench.get('dataset', 'N/A')}")
        print(f"     서브셋: {bench.get('subset', 'N/A')}")
        print(f"     샘플 수: {bench.get('sample_size', 'N/A')}")
        print(f"     유형: {bench.get('type', 'generated_answer')}")

    run_eval_script = PROJECT_ROOT / "scripts" / "run_eval.py"
    if run_eval_script.exists():
        cmd = [
            sys.executable, str(run_eval_script),
            "--track", "retention",
            "--config", "configs/eval.yaml",
        ]
        print(f"\n  실행 명령: {' '.join(cmd)}")

        result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))

        if result.returncode == 0:
            print("\n✅ retention 평가 완료")
            print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
        else:
            print(f"\n❌ 실행 실패 (exit code: {result.returncode})")
            print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
    else:
        print("\n  수동 실행:")
        print("    python scripts/run_eval.py --track retention --config configs/eval.yaml")

    print("\n  ℹ️  retention_delta_pp = 100 × (adapted_accuracy - base_accuracy)")
    print("     단일 벤치마크의 작은 샘플로는 망각 제거를 입증할 수 없습니다.")

In [ ]:
"""View per-task results."""

import json

print("=" * 70)
print("📋 태스크별 결과 보기")
print("=" * 70)

results_dir = output_dir

# Find result files
result_files = sorted(results_dir.glob("**/*.json"))
jsonl_files = sorted(results_dir.glob("**/*.jsonl"))

if not result_files and not jsonl_files:
    print("⚠️  결과 파일을 찾을 수 없습니다.")
    print(f"   결과 디렉토리: {results_dir}")
    print("   위의 평가 트랙을 먼저 실행하세요.")
else:
    print(f"결과 파일: {len(result_files)} JSON, {len(jsonl_files)} JSONL\n")

    # Load and display summary files
    for rf in result_files:
        if "summary" in rf.name or rf.name == "results.json":
            print(f"\n--- {rf.relative_to(results_dir)} ---")
            try:
                with open(rf) as f:
                    data = json.load(f)

                if isinstance(data, dict):
                    for key, value in data.items():
                        if isinstance(value, (int, float, str, bool)):
                            print(f"  {key}: {value}")
                        elif isinstance(value, dict) and len(value) <= 10:
                            print(f"  {key}:")
                            for k, v in value.items():
                                print(f"    {k}: {v}")
            except Exception as exc:
                print(f"  (로드 실패: {exc})")

    # Show per-task JSONL
    for jf in jsonl_files[:3]:
        print(f"\n--- {jf.relative_to(results_dir)} ---")
        try:
            with open(jf) as f:
                lines = f.readlines()[:5]
            for line in lines:
                rec = json.loads(line)
                task_id = rec.get("task_id", "?")
                success = rec.get("success", "?")
                error = rec.get("error", "")
                status = "✅" if success else "❌"
                print(f"  {status} {task_id}: success={success}", end="")
                if error:
                    print(f" error={error[:60]}")
                else:
                    print()
            if len(lines) >= 5:
                print(f"  ... (총 {sum(1 for _ in open(jf))}행)")
        except Exception as exc:
            print(f"  (로드 실패: {exc})")

In [ ]:
"""Log all evaluation results to MLflow."""

print("=" * 70)
print("📦 MLflow 평가 결과 기록")
print("=" * 70)

mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
experiment_name = os.environ.get("MLFLOW_EXPERIMENT_EVAL", "rhoai-model-training-lab-evaluation")

if not mlflow_uri:
    print("⚠️  MLFLOW_TRACKING_URI 미설정")
    print("   로컬 결과는 저장되었습니다.")
    print(f"   결과 경로: {output_dir}")
    print("\n   수동 업로드:")
    print("   python scripts/log_eval_results.py --results-dir", str(output_dir))
else:
    # Use log_eval_results script if available
    log_script = PROJECT_ROOT / "scripts" / "log_eval_results.py"
    if log_script.exists():
        cmd = [
            sys.executable, str(log_script),
            "--results-dir", str(output_dir),
        ]
        print(f"  실행: {' '.join(cmd)}")
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))

        if result.returncode == 0:
            print("\n✅ MLflow 기록 완료")
            print(result.stdout[-300:] if len(result.stdout) > 300 else result.stdout)
        else:
            print(f"\n❌ MLflow 기록 실패 (exit code: {result.returncode})")
            print(result.stderr[-300:] if len(result.stderr) > 300 else result.stderr)
    else:
        # Direct MLflow logging
        try:
            import mlflow

            mlflow.set_tracking_uri(mlflow_uri)
            mlflow.set_experiment(experiment_name)

            # Log each result file
            for rf in result_files:
                with mlflow.start_run(run_name=rf.stem) as run:
                    try:
                        with open(rf) as f:
                            data = json.load(f)
                        if isinstance(data, dict):
                            for k, v in data.items():
                                if isinstance(v, (int, float)):
                                    mlflow.log_metric(k, v)
                                elif isinstance(v, str):
                                    mlflow.log_param(k, v[:250])
                        mlflow.log_artifact(str(rf))
                        print(f"  ✅ {rf.name} → run_id: {run.info.run_id}")
                    except Exception as exc:
                        print(f"  ❌ {rf.name}: {exc}")

            print("\n✅ MLflow 기록 완료")
        except Exception as exc:
            print(f"\n❌ MLflow 기록 실패: {exc}")
            print("  로컬 결과는 보존되었습니다.")

print("\n다음 단계:")
print("  📓 08_compare_results.ipynb — 실험 결과 비교 분석")